In [1]:
from azure.identity import DefaultAzureCredential, ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient, FileSystemClient
from azure.keyvault.secrets import SecretClient
from azure.core.exceptions import ResourceNotFoundError

import os
import io
import numpy as np
import matplotlib.pyplot as plt
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrowfs_adlgen2 as pa_adl
import pickle
import json
import duckdb

from deltalake import DeltaTable, write_deltalake
from dotenv import load_dotenv
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import (
    SMOTE,
    SMOTENC,
    SMOTEN,
    SVMSMOTE,
    KMeansSMOTE,
    BorderlineSMOTE,
    RandomOverSampler,
    ADASYN,
)

from functools import reduce
from concurrent.futures import ThreadPoolExecutor
from utilities.loaders import save_model

%load_ext autoreload
%autoreload 2

c:\Users\LARRY\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


# local

In [2]:
DATA_DIR = "../../include/data"

In [3]:
# local
SILVER_FOLDER_NAME = "silver"
SUB_FOLDER_NAME = "stage-03"
SILVER_DATA_DIR = os.path.join("{DATA_DIR}", "{FOLDER_NAME}", "{SUB_FOLDER_NAME}").replace("\\", "/")
SILVER_DATA_DIR

'{DATA_DIR}/{FOLDER_NAME}/{SUB_FOLDER_NAME}'

# load credentials for cloud

In [4]:
# Retrieve credentials from environment variables
# this is strictly used only in development
# load env variables
env_dir = Path('../../').resolve()
load_dotenv(os.path.join(env_dir, '.env'))

True

In [5]:
storage_account_name = os.environ.get("STORAGE_ACCOUNT_NAME")
storage_account_key = os.environ.get("STORAGE_ACCOUNT_KEY")
storage_account_conn_str = os.environ.get("STORAGE_ACCOUNT_CONN_STR")

In [10]:
# cloud
URL = "abfss://{FOLDER_NAME}@sgppipelinesa.dfs.core.windows.net"
# URL = "{FOLDER_NAME}"
SILVER_FOLDER_NAME = "sgppipelinesa-silver"
SUB_FOLDER_NAME = "stage-03"
SILVER_DATA_DIR = os.path.join(URL, "{SUB_FOLDER_NAME}").replace("\\", "/")
SILVER_DATA_DIR

'abfss://{FOLDER_NAME}@sgppipelinesa.dfs.core.windows.net/{SUB_FOLDER_NAME}'

# this client is for saving .pkl, .json files to ADL2

In [5]:
# cloud
# create client with generated sas token
datalake_service_client = DataLakeServiceClient(
    account_url=f"https://{storage_account_name}.dfs.core.windows.net", 
    credential=storage_account_key
)

# retrieves file system client/container client 
# to retrieve datalake client
misc_container_client = datalake_service_client.get_file_system_client(f"{storage_account_name}-miscellaneous")

# this client is for saving pyarrow tables to ADL2 

In [6]:
handler = pa_adl.AccountHandler.from_account_name(storage_account_name, credential=storage_account_key)
fs = pa.fs.PyFileSystem(handler)

# Run if the following if spark cluster is not available to augment signal features in a distributed manner

In [6]:
conn = duckdb.connect()

In [7]:
# for cloud only
# installing dependencies and creating secrets object
conn.sql(f"""INSTALL azure""")
conn.sql(f"""LOAD azure""")
conn.sql(f"""
    CREATE OR REPLACE SECRET az_sgp (
        TYPE azure,
        CONNECTION_STRING '{storage_account_conn_str}'
    );
""")
# the is required if this notebook is run in linux environment
# like airflow container
conn.sql("SET azure_transport_option_type = 'curl'")

In [9]:
# # cloud
# data_path = os.path.join(
#     SILVER_DATA_DIR.format(
#         FOLDER_NAME=SILVER_FOLDER_NAME,
#         SUB_FOLDER_NAME="stage-03"
#     ),
#     "data"
# ).replace("\\", "/")
# data_path

# local
data_path = os.path.join(
    SILVER_DATA_DIR.format(
        DATA_DIR=DATA_DIR,
        FOLDER_NAME=SILVER_FOLDER_NAME,
        SUB_FOLDER_NAME="stage-03"
    ),
    "data"
).replace("\\", "/")
data_path

'../../include/data/silver/stage-03/data'

In [13]:
conn.sql(f"""
    CREATE OR REPLACE TEMPORARY TABLE signal_gender_data AS (
        SELECT
            * EXCLUDE(file_name, filename), 
            filename AS file_name 
        FROM delta_scan('{data_path}', filename=True)
    );
""")

In [ ]:
conn.sql("""
    SELECT *
    FROM signal_gender_data
    -- WHERE subject_id = '1028-20100710-hne'
    WHERE subject_id = 'Coren-20141121-pxp'   
""")

┌────────────────────┬─────────────────────┬────────────────────────────┬─────────────────────┬──────────────────────┬────────────────────┬───────────────────────┬─────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────────┬────────────────────────┬──────────────────────┬──────────────────────┬───────────────────┬────────────────────────┬──────────────────────┬─────────────────────────┬─────────────────────┬────────────────────┬────────────────────┬───────────────────────┬────────────────────┬────────────────────┬───────────────────────┬────────────────────────┬───────────────────┬────────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬───────────────────────┬──────────────────────┬────────────────────────────┬─────────────────────┬────────────────────┬────────────────────┬─────────────────────┬─────────────────────┬─────────────────────┬──────────────────────┬─────────────────────┬───

In [15]:
feature_infos = conn.sql("""
    SELECT 
        column_name, 
        data_type 
    FROM INFORMATION_SCHEMA.COLUMNS 
    WHERE table_name = 'signal_gender_data'
    AND column_name NOT IN (
        'subject_id', 
        'wav_file_row_number', 
        'file_name',
         
        'loaded_at',
        'label',
        'split',
        'feature_hash_key'
    )
""").fetchall()
feature_cols, _ = list(zip(*feature_infos))
feature_cols

('freq_kurt',
 'freq_skew',
 'freq_entropy',
 'freq_mean',
 'freq_median',
 'freq_mode',
 'freq_min',
 'freq_max',
 'freq_var',
 'freq_stddev',
 'freq_first_quart',
 'freq_third_quart',
 'freq_range',
 'freq_inter_quart_range',
 'zcr',
 'poly_feat_1',
 'poly_feat_2',
 'spec_cent',
 'spec_bw',
 'spec_flat',
 'spec_roll',
 'mel_spec_mean',
 'mel_spec_median',
 'mel_spec_mode',
 'mel_spec_mode_cnt',
 'mel_spec_min',
 'mel_spec_max',
 'mel_spec_range',
 'mel_spec_var',
 'mel_spec_std',
 'mel_spec_first_quart',
 'mel_spec_third_quart',
 'mel_spec_inter_quart_range',
 'mel_spec_entropy',
 'mel_spec_kurt',
 'mel_spec_skew',
 'mel_spec_db_mean',
 'mel_spec_db_median',
 'mel_spec_db_mode',
 'mel_spec_db_mode_cnt',
 'mel_spec_db_min',
 'mel_spec_db_max',
 'mel_spec_db_range',
 'mel_spec_db_var',
 'mel_spec_db_std',
 'mel_spec_db_first_quart',
 'mel_spec_db_third_quart',
 'mel_spec_db_inter_quart_range',
 'mel_spec_db_entropy',
 'mel_spec_db_kurt',
 'mel_spec_db_skew',
 'mfcc_mean',
 'mfcc_median

In [16]:
len(feature_cols)

81

# Convert the duckdb tables to pandas then to numpy to be passed as input to `SMOTE` method that will allow us to augment and oversample our minority class in each split

In [17]:
train_input = conn.sql(f"SELECT {', '.join(feature_cols)} FROM signal_gender_data WHERE split = 'train'").to_df().to_numpy()
train_input

array([[ 3.11519709, -1.65185075,  2.        , ...,  1.93199472,
         0.20256986,  0.95898564],
       [ 3.11519709,  1.72741621,  1.5849625 , ...,  1.91525464,
         1.26743854,  1.66333802],
       [ 3.11519709,  0.03778273,  1.        , ...,  1.90329172,
        -0.72900488,  0.8689818 ],
       ...,
       [-3.72920326, -1.72814502,  1.5849625 , ...,  1.90520498,
         0.58096618,  1.42366841],
       [-3.72920326, -0.82438092,  1.        , ...,  1.90675813,
         2.05505834,  1.98945307],
       [-3.72920326, -0.82438092,  0.        , ...,  1.90299625,
         1.53286225,  1.76148863]])

In [18]:
train_output = conn.sql(f"""
    SELECT 
        CASE 
            WHEN label = 'female' THEN 1
            ELSE 0
        END AS label_encoded 
    FROM signal_gender_data WHERE split = 'train'
""").to_df().to_numpy().ravel()
train_output

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])

In [19]:
val_input = conn.sql(f"SELECT {', '.join(feature_cols)} FROM signal_gender_data WHERE split = 'val'").to_df().to_numpy()
val_input

array([[ 0.        , -1.17595562,  1.5849625 , ...,  1.92604417,
        -0.9662606 ,  0.31208172],
       [ 0.        , -1.17595562,  1.        , ...,  1.93051586,
        -1.18408827,  0.23659653],
       [ 0.        , -1.17595562,  0.        , ...,  1.9382972 ,
        -1.19208783, -0.19750891],
       ...,
       [ 0.        ,  1.71584139,  1.5849625 , ...,  1.90312642,
         1.68221133,  1.83084414],
       [ 0.        ,  1.71584139,  1.        , ...,  1.90054198,
         1.61219396,  1.79351084],
       [ 0.        ,  1.71584139,  0.        , ...,  1.90659021,
        -0.44867096,  1.05832693]])

In [20]:
val_output = conn.sql(f"""
    SELECT 
        CASE 
            WHEN label = 'female' THEN 1
            ELSE 0
        END AS label_encoded 
    FROM signal_gender_data WHERE split = 'val'
""").to_df().to_numpy().ravel()
val_output

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])

In [21]:
test_input = conn.sql(f"SELECT {', '.join(feature_cols)} FROM signal_gender_data WHERE split = 'test'").to_df().to_numpy()
test_input

array([[ 0.        ,  0.24187871,  1.5849625 , ...,  1.92725603,
        -0.67126032,  0.83940795],
       [ 0.        ,  0.24187871,  1.        , ...,  1.89101055,
        -0.86661779,  0.96724162],
       [ 0.        ,  0.24187871,  0.        , ...,  1.89762522,
        -0.70249051,  0.85320745],
       ...,
       [ 4.2704093 ,  1.70469871,  1.5849625 , ...,  1.88453872,
        -0.86077941,  0.99661433],
       [ 4.2704093 ,  1.93524641,  1.        , ...,  1.91432684,
        -0.96691038,  0.84950277],
       [ 4.2704093 ,  1.93524641,  0.        , ...,  1.93569365,
         1.73096659,  1.82951828]])

In [22]:
test_output = conn.sql(f"""
    SELECT 
        CASE 
            WHEN label = 'female' THEN 1
            ELSE 0
        END AS label_encoded 
    FROM signal_gender_data WHERE split = 'test'
""").to_df().to_numpy().ravel()
test_output

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])

In [23]:
scaler = StandardScaler()

In [24]:
np.where(np.isinf(train_input))

(array([], dtype=int64), array([], dtype=int64))

In [25]:
train_input_sc = scaler.fit_transform(train_input)
val_input_sc = scaler.transform(val_input)
test_input_sc = scaler.transform(test_input)

In [26]:
# make save the sacler to the miscellaneous container or 
# bucket for later inference
scaler

,copy,True
,with_mean,True
,with_std,True


# Save standard scaler to miscellaneous container storage

In [28]:
# # cloud
# URL = "abfss://{FOLDER_NAME}@sgppipelinesa.dfs.core.windows.net"
# URL = "{FOLDER_NAME}"
# MISCELLANEOUS_FOLDER_NAME = "sgppipelinesa-miscellaneous"

# local
MISCELLANEOUS_FOLDER_NAME = "miscellaneous"
MISCELLANEOUS_DATA_DIR = os.path.join("{DATA_DIR}", "{FOLDER_NAME}").replace("\\", "/")
MISCELLANEOUS_DATA_DIR

'{DATA_DIR}/{FOLDER_NAME}'

In [ ]:
# # cloud write
# def write_model_to_fs(container_client: FileSystemClient, model, folder_name: str="", object_name: str="default"):
#     """
#     writes model to azure blob file system/container
#     """
    
#     # serialize model
#     model_ds = pickle.dumps(model)
    
#     model_client = container_client.get_file_client(os.path.join(folder_name, object_name).replace("\\", "/"))
#     model_client.upload_data(model_ds, overwrite=True)

In [ ]:
# # cloud write
# write_model_to_fs(misc_container_client, scaler, object_name="scaler.pkl")

In [29]:
# local save dir
SAVE_DIR = MISCELLANEOUS_DATA_DIR.format(
    DATA_DIR=DATA_DIR,
    FOLDER_NAME=MISCELLANEOUS_FOLDER_NAME
)

In [30]:
# local save path
scaler_path = os.path.join(SAVE_DIR, "scaler.pkl").replace("\\", "/")
scaler_path

'../../include/data/miscellaneous/scaler.pkl'

In [31]:
# locally saving scaler
save_model(scaler, path=scaler_path)

# Define `SMOTE` to augment our signal features

In [32]:
# oversampling the train dataset using SMOTE
smt = SMOTE()

In [33]:
train_input_sc_sm, train_output_sm = smt.fit_resample(train_input_sc, train_output)
val_input_sc_sm, val_output_sm = smt.fit_resample(val_input_sc, val_output)
test_input_sc_sm, test_output_sm = smt.fit_resample(test_input_sc, test_output)

In [34]:
np.unique(test_output_sm, return_counts=True)

(array([0, 1]), array([73, 73], dtype=int64))

In [35]:
train_output_sm.reshape(-1, 1).shape

(352, 1)

# reconcatenate the input and output matrix and array of each split this is so when tables are saved to parquet if ever there is partitioning the rows in input and output are not reshuffled

In [36]:
train_input_sc_sm.shape

(352, 81)

In [37]:
# we need to reshape the output array as this needs to be
# in a matrix shape in order to concatenate to our input matrix
train_data_sc_sm = np.concatenate([train_input_sc_sm, train_output_sm.reshape(-1, 1)], axis=1)
val_data_sc_sm = np.concatenate([val_input_sc_sm, val_output_sm.reshape(-1, 1)], axis=1)
test_data_sc_sm = np.concatenate([test_input_sc_sm, test_output_sm.reshape(-1, 1)], axis=1)
train_data_sc_sm.shape, val_data_sc_sm.shape, test_data_sc_sm.shape

((352, 82), (146, 82), (146, 82))

# Convert numpy array of the augmented signals as well as labels to pyarrow tables to save as parquet

In [38]:
# because we concatenated our input and output
# labels again we need to include in our feature
# cols list this labels column again when we save
# our newly concatenated table as parquet again
cols = list(feature_cols) + ["label"]

In [39]:
train_data_sc_sm_table = pa.table({feat_col: pa.array(train_data_sc_sm[:, i]) for i, feat_col in enumerate(cols)})
val_data_sc_sm_table = pa.table({feat_col: pa.array(val_data_sc_sm[:, i]) for i, feat_col in enumerate(cols)})
test_data_sc_sm_table = pa.table({feat_col: pa.array(test_data_sc_sm[:, i]) for i, feat_col in enumerate(cols)})

In [40]:
train_data_sc_sm_table.shape

(352, 82)

In [41]:
val_data_sc_sm_table.shape

(146, 82)

In [42]:
test_data_sc_sm_table.shape

(146, 82)

# save the final scaled and augmented features to the gold layer

In [43]:
# local
GOLD_FOLDER_NAME = "gold"
GOLD_DATA_DIR = os.path.join("{DATA_DIR}", "{FOLDER_NAME}").replace("\\", "/")
SAVE_DIR = GOLD_DATA_DIR.format(
    DATA_DIR=DATA_DIR,
    FOLDER_NAME=GOLD_FOLDER_NAME,
)
SAVE_DIR

# # cloud
# GOLD_FOLDER_NAME = "sgppipelinesa-gold"
# GOLD_DATA_DIR = os.path.join("{FOLDER_NAME}").replace("\\", "/")
# SAVE_DIR = GOLD_DATA_DIR.format(
#     FOLDER_NAME=GOLD_FOLDER_NAME,
# )
# SAVE_DIR

'../../include/data/gold'

In [44]:
train_data_sc_sm_table_path = os.path.join(SAVE_DIR, "train_data_sc_sm.parquet").replace("\\", "/")
val_data_sc_sm_table_path = os.path.join(SAVE_DIR, "val_data_sc_sm.parquet").replace("\\", "/")
test_data_sc_sm_table_path = os.path.join(SAVE_DIR, "test_data_sc_sm.parquet").replace("\\", "/")
train_data_sc_sm_table_path, val_data_sc_sm_table_path, test_data_sc_sm_table_path

('../../include/data/gold/train_data_sc_sm.parquet',
 '../../include/data/gold/val_data_sc_sm.parquet',
 '../../include/data/gold/test_data_sc_sm.parquet')

In [46]:
# # cloud
# pq.write_table(train_data_sc_sm_table, train_data_sc_sm_table_path, filesystem=fs)
# pq.write_table(val_data_sc_sm_table, val_data_sc_sm_table_path, filesystem=fs)
# pq.write_table(test_data_sc_sm_table, test_data_sc_sm_table_path, filesystem=fs)

# local
os.makedirs(SAVE_DIR, exist_ok=True)
pq.write_table(train_data_sc_sm_table, train_data_sc_sm_table_path)
pq.write_table(val_data_sc_sm_table, val_data_sc_sm_table_path)
pq.write_table(test_data_sc_sm_table, test_data_sc_sm_table_path)